# Construindo a Arquitetura da CNN 

![Extracao de características](extracao_caracteristicas.png)

## 1. Importando as bibliotecas

In [1]:
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Convolution2D
from keras.layers import MaxPooling2D
from keras.layers import Flatten
from keras.layers import Dense
from keras.layers import Dropout
from keras import utils
import numpy as np

## 2. Aquisição dos dados

In [3]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

11490434/11490434 [==============================] - 1s 0us/step


In [4]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(60000, 28, 28)
(60000,)
(10000, 28, 28)
(10000,)


## 3. Pré-processamento

In [5]:
X_train = X_train / 255.
X_test = X_test / 255.
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], X_train.shape[2], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], X_test.shape[2], 1)
y_train = utils.to_categorical(y_train) #8 -> 0 0 0 0 0 0 0 0 1 0
y_test = utils.to_categorical(y_test) #3 -> 0 0 0 1 0 0 0 0 0 0

In [6]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(60000, 28, 28, 1)
(60000, 10)
(10000, 28, 28, 1)
(10000, 10)


## 4. Arquitetura da CNN

![Arquitetura CNN](cnn_arquitetura_basica.png)

In [7]:
# Inicializando a CNN
classifier = Sequential()

#Camada de convolução
classifier.add(Convolution2D(32, kernel_size=(3,3), input_shape = (28, 28,1), activation = 'relu', padding='same', name = 'conv_1'))

#Camada de pooling
classifier.add(MaxPooling2D(pool_size=(2,2), strides=(2, 2), padding='same', name = 'pool_1'))

#Segunda camada convolucional
classifier.add(Convolution2D(64, kernel_size=(3,3), activation = 'relu', padding='same', name = 'conv_2'))


#Segunda camada de pooling
classifier.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding='same', name = 'pool_2'))


#Vetorizando os mapas de características do último pooling (camada de entrada)
classifier.add(Flatten())

#Dropout
classifier.add(Dropout(0.5))

#Camada totalmente conectada ou oculta
classifier.add(Dense(activation='relu', units=128, name = 'dense_1'))


#Camada de saída
classifier.add(Dense(activation='softmax', units=10,  name = 'classification'))

In [8]:
classifier.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv_1 (Conv2D)             (None, 28, 28, 32)        320       
                                                                 
 pool_1 (MaxPooling2D)       (None, 14, 14, 32)        0         
                                                                 
 conv_2 (Conv2D)             (None, 14, 14, 64)        18496     
                                                                 
 dropout (Dropout)           (None, 14, 14, 64)        0         
                                                                 
 pool_2 (MaxPooling2D)       (None, 7, 7, 64)          0         
                                                                 
 flatten (Flatten)           (None, 3136)              0         
                                                                 
 dropout_1 (Dropout)         (None, 3136)              0

## 5. Treinando o modelo

In [13]:
#Parâmetros de treinamento
epochs = 5
batch_size = 50
validation_split=0.1

In [14]:
print(54000/50)

1080.0


In [15]:
classifier.compile(optimizer = 'sgd', loss= 'categorical_crossentropy', metrics=['accuracy'])

checkpoint = keras.callbacks.ModelCheckpoint('best_model.h5', monitor='val_loss', verbose=1, save_best_only=True, mode='auto', save_freq='epoch') 
earlystop = keras.callbacks.EarlyStopping(patience=15)

In [19]:
classifier.fit(X_train, y_train, validation_split=validation_split, batch_size=batch_size, epochs=epochs, callbacks=[checkpoint,earlystop], verbose=1)

Epoch 1/5
1079/1080 [============================>.] - ETA: 0s - loss: 0.0814 - accuracy: 0.9740
Epoch 1: val_loss improved from 0.05451 to 0.05216, saving model to best_model.h5
1080/1080 [==============================] - 27s 25ms/step - loss: 0.0815 - accuracy: 0.9740 - val_loss: 0.0522 - val_accuracy: 0.9862
Epoch 2/5
1079/1080 [============================>.] - ETA: 0s - loss: 0.0775 - accuracy: 0.9758
Epoch 2: val_loss improved from 0.05216 to 0.04984, saving model to best_model.h5
1080/1080 [==============================] - 28s 26ms/step - loss: 0.0775 - accuracy: 0.9758 - val_loss: 0.0498 - val_accuracy: 0.9872
Epoch 3/5
1079/1080 [============================>.] - ETA: 0s - loss: 0.0736 - accuracy: 0.9774
Epoch 3: val_loss improved from 0.04984 to 0.04938, saving model to best_model.h5
1080/1080 [==============================] - 28s 26ms/step - loss: 0.0735 - accuracy: 0.9774 - val_loss: 0.0494 - val_accuracy: 0.9883
Epoch 4/5
1079/1080 [============================>.] - ETA

## 6. Avaliando o modelo

In [20]:
best_model = keras.models.load_model("best_model.h5")

In [21]:
score = best_model.evaluate(X_test, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])

Test loss: 0.04152359813451767
Test accuracy: 0.9868000149726868


𝐴𝑡𝑖𝑣𝑖𝑑𝑎𝑑𝑒:  Treinar e Avaliar a arquitetura adaptada para outro conjunto de dados.